# Phase 2-1: exact_skill_score 검증

Notion 스펙의 `exact_skill_score` + `capability_score` 구현을 검증한다.

### 검증 항목
1. `calc_exact_skill_score` — 스킬 매칭 비율 계산
2. `capability_score` = 0.8×exact + 0.2×dense 결합
3. `HybridSearcher` + `exact_skill_scores` 주입 검색
4. exact 적용 전(BM25+RRF) vs 후(exact+dense) 랭킹 변화 비교

In [1]:
from _bootstrap import setup_project_path
setup_project_path()

WindowsPath('C:/Users/mk.jang/Desktop/TLC/08_TSM/retrieval-lab')

In [2]:
import numpy as np

from embedding_retrieval.config import RetrievalConfig
from embedding_retrieval.factory import create_embeddings
from embedding_retrieval.stores.dual_upstash import DualUpstashStore
from embedding_retrieval.scenarios.sample_data import SAMPLE_ENGINEER_PROFILES
from embedding_retrieval.search.exact_skill import (
    calc_exact_skill_score,
    calc_capability_score,
    parse_skills_from_capability,
)

config = RetrievalConfig.from_env()
embeddings = create_embeddings(config)
profiles = SAMPLE_ENGINEER_PROFILES

dual_store = DualUpstashStore(
    embeddings=embeddings,
    url=config.vector_store_kwargs["url"],
    token=config.vector_store_kwargs["token"],
)

print(f"엔지니어 {len(profiles)}명 로드")
print("DualUpstashStore 초기화 완료")

엔지니어 100명 로드
DualUpstashStore 초기화 완료


## 1. calc_exact_skill_score 단독 테스트

required_skills 기준으로 각 엔지니어의 매칭 비율을 계산한다.

In [3]:
# 시나리오별 required_skills 정의
scenarios = [
    {
        "name": "Java/Spring 백엔드",
        "required_skills": ["Java", "Spring Boot", "PostgreSQL"],
        "capability_query": "Java / Spring Boot / PostgreSQL",
        "experience_query": "[프로젝트] 제조업 ERP 시스템 개발\n[포지션] 백엔드 개발자\n[기타요건] 제조업 경험 우대",
        "cap_weight": 0.6,
        "exp_weight": 0.4,
    },
    {
        "name": "React 프론트엔드",
        "required_skills": ["React", "TypeScript"],
        "capability_query": "React / TypeScript",
        "experience_query": "[프로젝트] 이커머스 프론트엔드 개발\n[포지션] 프론트엔드 개발자\n[기타요건] 대시보드 경험 우대",
        "cap_weight": 0.7,
        "exp_weight": 0.3,
    },
    {
        "name": "Python/FastAPI 백엔드",
        "required_skills": ["Python", "FastAPI", "Docker"],
        "capability_query": "Python / FastAPI / Docker",
        "experience_query": "[프로젝트] MSA 플랫폼 구축\n[포지션] 백엔드 개발자\n[기타요건] Kubernetes 경험 우대",
        "cap_weight": 0.5,
        "exp_weight": 0.5,
    },
]

# 각 시나리오에서 엔지니어별 exact_skill_score 계산
for sc in scenarios:
    print(f"\n=== {sc['name']} (required: {sc['required_skills']}) ===")
    print(f"  {'engineer_id':<12} {'skills':<45} {'exact':>6}")
    print(f"  {'-'*67}")
    results = []
    for p in profiles:
        eng_skills = parse_skills_from_capability(p.capability_text)
        score = calc_exact_skill_score(sc["required_skills"], eng_skills)
        results.append((p.engineer_id, eng_skills, score))
    # score 높은 순 top-5만 출력
    for eid, skills, score in sorted(results, key=lambda x: -x[2])[:5]:
        skills_str = " / ".join(skills[:4]) + (" ..." if len(skills) > 4 else "")
        print(f"  {eid:<12} {skills_str:<45} {score:>6.2f}")


=== Java/Spring 백엔드 (required: ['Java', 'Spring Boot', 'PostgreSQL']) ===
  engineer_id  skills                                         exact
  -------------------------------------------------------------------
  eng-001      Java / Spring Boot / Spring Batch / PostgreSQL ...   1.00
  eng-013      Java / Spring Boot / Spring Batch / PostgreSQL ...   1.00
  eng-025      Java / Spring Boot / Spring Batch / PostgreSQL ...   1.00
  eng-037      Java / Spring Boot / Spring Batch / PostgreSQL ...   1.00
  eng-049      Java / Spring Boot / Spring Batch / PostgreSQL ...   1.00

=== React 프론트엔드 (required: ['React', 'TypeScript']) ===
  engineer_id  skills                                         exact
  -------------------------------------------------------------------
  eng-006      React / TypeScript / Next.js / TanStack Query ...   1.00
  eng-007      React / Vue.js / TypeScript / Vite ...          1.00
  eng-018      React / TypeScript / Next.js / TanStack Query ...   1.00
  eng-019      

## 2. capability_score = 0.8×exact + 0.2×dense

Upstash에서 저장된 dense 벡터를 fetch하고 쿼리와 cosine 유사도를 계산한 뒤,
`calc_capability_score`로 결합한다.

In [5]:
# Upstash에서 capability 벡터 fetch — 임베딩 API 재호출 없음
print("capability 벡터 fetch 중...")
cap_vec_map = dual_store.fetch_cap_vectors()  # dict[engineer_id → np.ndarray]
print(f"fetch 완료: {len(cap_vec_map)}명\n")

sc = scenarios[0]  # Java/Spring 백엔드 시나리오
print(f"=== {sc['name']} — capability_score 결합 ===")
print(f"  {'engineer_id':<12} {'exact':>6} {'dense':>6} {'cap_score':>10}  (0.8×exact + 0.2×dense)")
print(f"  {'-'*55}")

# 쿼리 임베딩 (1번만 호출)
q_vec = np.array(embeddings.embed_query(sc["capability_query"]))
q_norm = q_vec / np.linalg.norm(q_vec)

rows = []
for p in profiles:
    eng_skills = parse_skills_from_capability(p.capability_text)
    exact = calc_exact_skill_score(sc["required_skills"], eng_skills)

    cap_vec = cap_vec_map.get(p.engineer_id)
    if cap_vec is None:
        continue
    cap_norm = cap_vec / np.linalg.norm(cap_vec)
    dense = float(cap_norm @ q_norm)

    cap_score = calc_capability_score(exact, dense)
    rows.append((p.engineer_id, exact, dense, cap_score))

for eid, exact, dense, cap_score in sorted(rows, key=lambda x: -x[3])[:10]:
    print(f"  {eid:<12} {exact:>6.2f} {dense:>6.4f} {cap_score:>10.4f}")

capability 벡터 fetch 중...


ConnectError: [Errno 11001] getaddrinfo failed

## 3. HybridSearcher + exact_skill_scores 주입 검색

`exact_skill_scores` dict를 미리 계산해서 `HybridSearcher.search()`에 주입한다.

In [ ]:
from embedding_retrieval.search.bm25 import BM25Index
from embedding_retrieval.search.hybrid import HybridSearcher

# BM25 인덱스 빌드
cap_bm25 = BM25Index()
exp_bm25 = BM25Index()
for p in profiles:
    cap_bm25.add(p.engineer_id, p.capability_text)
    exp_bm25.add(p.engineer_id, p.experience_text)
cap_bm25.build()
exp_bm25.build()

searcher = HybridSearcher(
    store=dual_store,
    cap_bm25=cap_bm25,
    exp_bm25=exp_bm25,
    profiles=profiles,
)
print("HybridSearcher 초기화 완료")

# 시나리오별 exact_skill_scores 계산 후 검색
for sc in scenarios:
    # 전체 엔지니어 exact_skill_score 사전 계산
    exact_scores = {
        p.engineer_id: calc_exact_skill_score(
            sc["required_skills"],
            parse_skills_from_capability(p.capability_text),
        )
        for p in profiles
    }

    results = searcher.search(
        capability_query=sc["capability_query"],
        experience_query=sc["experience_query"],
        weights=(sc["cap_weight"], sc["exp_weight"]),
        top_k=5,
        only_available=True,
        only_full_time=True,
        exact_skill_scores=exact_scores,
    )

    print(f"\n=== {sc['name']} (cap_w={sc['cap_weight']}, exp_w={sc['exp_weight']}) ===")
    print(f"  {'rank':<5} {'engineer_id':<12} {'exact':>6} {'dense_cap':>10} {'cap_score':>10} {'exp_score':>10} {'final':>8}")
    print(f"  {'-'*70}")
    for c in results:
        b = c.score_breakdown
        print(f"  {c.rank:<5} {c.engineer_id:<12} {b.exact_skill_score:>6.2f} {b.dense_capability_score:>10.4f} {b.capability_score:>10.4f} {b.experience_score:>10.4f} {b.final_score:>8.4f}")

## 4. exact 적용 전(BM25+RRF) vs 후(exact+dense) 랭킹 변화

동일 쿼리에 대해 두 방식의 top-5 랭킹을 나란히 비교한다.

In [ ]:
sc = scenarios[0]  # Java/Spring 백엔드

exact_scores = {
    p.engineer_id: calc_exact_skill_score(
        sc["required_skills"],
        parse_skills_from_capability(p.capability_text),
    )
    for p in profiles
}

# Before: BM25+RRF (exact_skill_scores=None)
before = searcher.search(
    capability_query=sc["capability_query"],
    experience_query=sc["experience_query"],
    weights=(sc["cap_weight"], sc["exp_weight"]),
    top_k=5,
    exact_skill_scores=None,
)

# After: exact + dense
after = searcher.search(
    capability_query=sc["capability_query"],
    experience_query=sc["experience_query"],
    weights=(sc["cap_weight"], sc["exp_weight"]),
    top_k=5,
    exact_skill_scores=exact_scores,
)

print(f"=== {sc['name']} — BM25+RRF vs exact+dense 랭킹 비교 ===")
print(f"  {'rank':<5} {'[BM25+RRF]':<20} {'score':>8}  |  {'[exact+dense]':<20} {'score':>8}  {'exact':>6}")
print(f"  {'-'*75}")
for b_c, a_c in zip(before, after):
    b_score = b_c.score_breakdown.final_score
    a_score = a_c.score_breakdown.final_score
    a_exact = a_c.score_breakdown.exact_skill_score
    moved = "" if b_c.engineer_id == a_c.engineer_id else " ←변동"
    print(f"  {b_c.rank:<5} {b_c.engineer_id:<20} {b_score:>8.4f}  |  {a_c.engineer_id:<20} {a_score:>8.4f}  {a_exact:>6.2f}{moved}")